In [ ]:
import anthropic
import json

from statistics import mean

from dotenv import load_dotenv
load_dotenv()

def add_user_messages(messages, message):
    messages.append({"role": "user", "content": message})

def add_assistant_messages(messages, message):
    messages.append({"role": "assistant", "content": message})
    
def chat(messages, system=None, temperature=1.0, stop_sequences=None):
    parameters = {
        "model": "claude-sonnet-4-6",
        "max_tokens": 512,
        "messages": messages,
        "temperature": temperature
    }
    
    if system:
        parameters["system"] = system
    
    if stop_sequences:
        parameters["stop_sequences"] = stop_sequences

    client = anthropic.Anthropic()
    response = client.messages.create(**parameters)
    return response.content[0].text
    

In [ ]:
import ast
import re

def validate_python_syntax(code):
    try:
        ast.parse(code)
        return 10
    except SyntaxError:
        return 0

def validate_json_syntax(code):
    try:
        json.loads(code)
        return 10
    except json.JSONDecodeError:
        return 0

def validate_regex_syntax(code):
    try:
        re.compile(code)
        return 10
    except re.error:
        return 0

In [ ]:
def grade_syntax(output, format):
    if format == "python":
        return validate_python_syntax(output)
    elif format == "json":
        return validate_json_syntax(output)
    elif format == "regex":
        return validate_regex_syntax(output)
    else:
        raise ValueError(f"Unknown format: {format}")

def grade_by_model(task, output):
    messages = []
    prompt = f"""
        You are an expert code reviewer. Evaluate this AI-generated solution.
        Please grade the following output for the task: 
        Task: 
        <task>
            {task["task"]}
        </task>

        Solution:
        <solution>
            {output}
        </solution>

        Provide your evaluation as JSON object:
        - grade: integer from 0 to 10, where 10 is the best grade.
        - comment: short explanation of the grade. no more than 32 chars. be concise

        Output should be without any markdown or code block, only JSON object. 
        Don't wrap output in any markdown. Don't use any backticks.

        Criteria for grading:
        <criteria>
            {task["solution_criteria"]}
        </criteria>
    """
    add_user_messages(messages, prompt)
    grade = chat(messages, system=""" 
                 You are an expert code reviewer. 
                 Output should be only JSON object with grade and comment. 
                 No markdown or code blocks. Don't wrap output in any markdown.
                 """)
    
    print(f"Grade response: {grade}")
    return json.loads(grade)

In [ ]:

def run_prompt(task):
    messages = []
    prompt = f"""
        Please solve the following task: {task["task"]}
        Provide only the code as output, without any explanations or markdown.
        Be consise and to the point. Don't include any comments in the code.
        Solution should be as short as possible while still being correct.
        Minify the code as much as possible. Don't use any unnecessary whitespace, such as newlines or spaces.
    """
    add_user_messages(messages, prompt)
    output = chat(messages)
    return output

def run_test_case(task):
    output = run_prompt(task)
    
    by_model = grade_by_model(task, output)
    by_syntax = grade_syntax(output, task["format"]) 

    result = {
        "task": task["task"],
        "output": output,
        "grade": by_model,
        "grade_syntax": by_syntax,
        "grade_total": (by_model["grade"] + by_syntax) / 2
    }

    return result

def run_eval(dataset):
    results = []
    for task in dataset:
        print(f"Running test case: {task}")
        result = run_test_case(task)
        print(f"Result: {result}")
        results.append(result)
    
    average_score = mean([result["grade_total"] for result in results])
    print(f"Average score: {average_score}")

    return results

In [ ]:
class PromptingEvaluator:
    def __init__(self, model="claude-sonnet-4-6", max_tokens=512, temperature=1.0):
        self.model = model
        self.max_tokens = max_tokens
        self.temperature = temperature
        self.client = anthropic.Anthropic()

    def _chat(self, messages, system=None, temperature=None):
        params = {
            "model": self.model,
            "max_tokens": self.max_tokens,
            "messages": messages,
            "temperature": temperature or self.temperature,
        }
        if system:
            params["system"] = system
        return self.client.messages.create(**params).content[0].text

    def run_prompt(self, task):
        messages = [{"role": "user", "content": f"""
            Please solve the following task: {task["task"]}
            Provide only the code as output, without any explanations or markdown.
            Be concise. No comments. Minify as much as possible.
        """}]
        return self._chat(messages)

    def grade_syntax(self, output, fmt):
        return grade_syntax(output, fmt)

    def grade_by_model(self, task, output):
        return grade_by_model(task, output)

    def run_test_case(self, task):
        output = self.run_prompt(task)
        by_model = self.grade_by_model(task, output)
        by_syntax = self.grade_syntax(output, task["format"])
        return {
            "task": task["task"],
            "output": output,
            "grade": by_model,
            "grade_syntax": by_syntax,
            "grade_total": (by_model["grade"] + by_syntax) / 2,
        }

    def run_eval(self, dataset):
        results = []
        for task in dataset:
            print(f"Running: {task['task'][:60]}...")
            result = self.run_test_case(task)
            print(f"  grade_total: {result['grade_total']}")
            results.append(result)
        avg = mean([r["grade_total"] for r in results])
        print(f"Average score: {avg:.2f}")
        return results

    def generate_dataset(self, instructions):
        messages = [{"role": "user", "content": instructions}]
        text = dataset(messages)
        return json.loads(text)

In [ ]:
with open("dataset.json", "r") as f:
    # dataset = json.loads(json.load(f))
    dataset = json.load(f)
results = run_eval(dataset)

print(json.dumps(results, indent=4))

In [ ]:

def dataset(messages):
    client = anthropic.Anthropic()
    with client.messages.stream(
        model="claude-haiku-4-5",
        max_tokens=1024,
        messages=messages,
        system="""
            Generate JSON array of tasks to measure prompt efficieny. 
            Tasks have to cover next areas: Python, Regex, AWS configuration.
            Solutions can be in next different formats only: Python, JSON, Regex.
            Each solution has to include 'format' field with format of the solution.
            Output should be without any markdown or code block.

            Example output:
            ```json
            [
                {
                    "task": "Description of task",
                    "format": "python",
                    "solution_criteria": "Key criteria for solution to be considered correct" 
                }
            ]
            ```

            Please generate 3 objects. 
            """,
    ) as stream:
        for response in stream.text_stream:
            pass
    return stream.get_final_message().content[0].text

In [ ]:
messages = []
add_user_messages(messages, """
                  Generate 2 different coding tasks of medium difficulty. 
                  Create maximum one Python task.
                  Don't include any explanations, only JSON array as output. 
                  No any intoduction or conclusion, only JSON array.
                  No backticks, code blocks or markdown, only JSON array.
                  Don't do: Dataset: [JSON Array], just output JSON array.
                  """)
output = dataset(messages)
print(f"Dataset: {output}")
with open("dataset.json", "w") as f:
    json.dump(json.loads(output), f, indent=4)